# Climate Forecasts

1. DATASETS: [Hugging Face: Howard Tsai | All](https://huggingface.co/Howard881010)
2. DATASETS: [Hugging Face: Howard Tsai | Climate-1 Day](https://huggingface.co/datasets/Howard881010/climate-1day)
3. REPO: [Multimodal_Forecasting](https://github.com/Rose-STL-Lab/Multimodal_Forecasting/tree/main)
    - Get data from here.


In [1]:
import os, sys

import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
# climate_dataset = os.path.join(base_data_path, 'others/', 'paper_mf/paper_mf_repo/data/climate/climate_2014_2023_final.csv')
# climate_dataset = os.path.join(base_data_path, 'others/', 'paper_mf/paper_mf_repo/data/climate/climate_2014_2023_final.csv')
climate_dataset = os.path.join(base_data_path, 'multimodal_forecast/climate_forecasts/climate_2014_2023_final.csv')

In [4]:
df = DataProcessing.load_from_file(climate_dataset)
df

,date,temp,precip,humidity,windspeed,text
0,2014-01-01,37.9,0.000,57.4,10.0,"Arctic air is expected to invade the upper Midwest, Great Lakes, Ohio Valley, and Northeast, bringing some of the coldest temperatures in recent memory. Significant cold outbreaks on January 3-4 and 5-8 could result in temperatures 20-30 degrees Fahrenheit below normal. Lows in New England may reach -10s to -20s, and areas in eastern Montana, Minnesota, and Wisconsin may experience lows at or below -20F. Colder locations could drop to -30s or -40s Fahrenheit. Moderate upslope snow is anticipated across east-facing slopes from Montana to Northeast Wyoming. The coldest day is projected around January 8, with temperatures below freezing in a large section of the northern United States, potentially marking the coldest air masses since 1993/1994.\n\nPrecipitation is expected from a frontal ..."
1,2014-01-02,35.9,0.161,82.0,11.3,"Frigid air will sweep into the Midwest, Great Lakes, Ohio Valley, Northeast, and Mid-Atlantic early next week. A deep vortex will lead to significant temperature drops of -10 to -20 degrees across much of the central and eastern U.S., with the Ohio Valley potentially seeing temperatures nearly 40 degrees below normal by Monday. Heavy snow is anticipated northwest of a surface low tracking from the Lower Mississippi Valley to Interior New England, accompanied by strong winds, particularly across the Great Lakes. A strong Arctic front will move across the Appalachians on Sunday. By mid-week, temperatures should begin to moderate in the eastern half of the nation. Increasing moisture will bring rounds of precipitation to the Pacific Northwest, Intermountain West, and Great Basin as initia..."
2,2014-01-03,22.4,0.015,54.1,30.5,"Another arctic plunge is expected for the northeast quarter of the US Monday and Tuesday, leading to temperatures 20 to 40 degrees Fahrenheit below normal. A deep upper-level system over eastern Canada will influence this frigid air, while a fresh north Pacific flow will bring cooler and unsettled weather to the Great Lakes and northeast. Light to moderate precipitation is forecasted across the west, with potential localized enhancement along the coast. As the week progresses, precipitation is expected to increase from the southern plains into the Ohio and Tennessee valleys, with rain in the south transitioning to snow/mix further north. The forecast maintains good continuity despite some discrepancies between models regarding the impacts of cold advection associated with the arctic in..."
3,2014-01-04,25.4,0.000,52.6,13.3,"20°F-40°F below normal temperatures are expected in the northeastern quarter of the U.S. for January 7-11, 2014, with Indiana, Ohio, and western Pennsylvania experiencing temperatures 20°F+ below normal. An arctic blast will affect these regions, while the southeast and mid-Atlantic areas will also experience lower temperatures. The forecast incorporates a blend of GFS and GEFS solutions, with a notable phase of energy in the northern stream beyond day 4."
4,2014-01-05,37.1,0.041,78.7,11.0,"Temperatures are expected to approach normal values, with possibilities of being above normal. A progressive weather pattern will dominate the lower 48 states. Significant precipitation and strong winds are anticipated for the Pacific Northwest due to cyclogenesis off the Oregon/Washington coast. In the Midwest and Plains, a system originating from the South Central Rockies will bring rain through the Ohio Valley, influenced by a surface ridge in the Northeast and Mid-Atlantic. Mid-level warm advection will contribute to this precipitation. The GFS model shows discrepancies in predicted conditions in the Mid-Missouri Valley, while Gulf of Mexico moisture is expected to affect the Southern Plains and Lower-to-Mid Missouri Valley. A cutoff low over the Rio Grande Valley will also enhance..."
...,...,...,...,...,...,...
3617,2023-11-27,43.0,0.004,62.0,20.4,"Moderate to heavy rainfall is expected from Thursday

In [7]:
df['text_len'] = df['text'].str.split().str.len()
df['text_len']

0       239
1       130
2       121
3        69
4       124
       ... 
3617    121
3618    107
3619    198
3620    204
3621    155
Name: text_len, Length: 3622, dtype: int64

In [8]:
df['text_len'].max()

np.int64(323)

In [ ]:
df['text'][0] = "Arctic air is expected to invade the upper Midwest, Great Lakes, Ohio Valley, and Northeast, bringing some of the coldest temperatures in recent memory. Significant cold outbreaks on January 3-4 and 5-8 could result in temperatures 20-30 degrees Fahrenheit below normal Lows in New England may reach -10s to -20s, and areas in eastern Montana, Minnesota, and Wisconsin may experience lows at or below -20F? Colder locations could drop to -30s or -40s Fahrenheit?"

df

In [ ]:
sfe = SpacyFeatureExtraction(df, 'text')
sfe

In [ ]:
# df = sfe.split_into_sentences(sentence_col_name='sentence')
# df

In [ ]:
df['label'] = 1
df

In [ ]:
save_path = os.path.join(base_data_path, 'multimodal_forecast', 'climate_forecasts')
DataProcessing.save_to_file(df, save_path, 'climate_forecasts', save_file_type='.csv')

> NEED TO DROP BAD SENTENCES, THEN RESAVE

In [ ]:
# DataProcessing.load_mf_climate_dataset(notebook_dir)

## Check bad sentences

In [ ]:
df[
    df["text"].str.contains(r"°F", regex=True, na=False)
][["text"]].head(10)

In [ ]:
# Return sentences that likely indicate bad sentence splits
bad_sentences = df[
    # Starts with dangling symbols or units (but not valid numbers)
    df["text"].str.match(r"^(°F|\+F|F\+|[^\w\d])", na=False)
    |
    # Ends mid temperature or numeric range expression
    df["text"].str.contains(r"(°F[-+]$|\d-\s*$)", regex=True, na=False)
]

bad_sentences[["text"]]

## Drop these, then save

In [ ]:
# Create a boolean mask for the bad rows
bad_mask = (
    df["text"].str.match(r"^(°F|\+F|F\+|[^\w\d])", na=False) |
    df["text"].str.contains(r"(°F[-+]$|\d-\s*$)", regex=True, na=False)
)

# Keep the good rows
df = df[~bad_mask]          # or df = df.loc[~bad_mask]
df

In [ ]:
save_path = os.path.join(base_data_path, 'multimodal_forecast', 'climate_forecasts')
DataProcessing.save_to_file(df, save_path, 'climate_forecasts', include_version=True, save_file_type='.csv')

In [ ]:
sfe = SpacyFeatureExtraction(df, 'text')
sfe

In [ ]:
sfe.sentence_embeddings_extraction(attach_to_df=True)

In [ ]:
df = sfe.split_into_sentences(sentence_col_name='sentence')
df

In [ ]:
df[
    df["text"].str.contains(r"°F", regex=True, na=False)
][["text"]]

In [ ]:
# Return sentences that likely indicate bad sentence splits
bad_sentences = df[
    # Starts with dangling symbols or units (but not valid numbers)
    df["text"].str.match(r"^(°F|\+F|F\+|[^\w\d])", na=False)
    |
    # Ends mid temperature or numeric range expression
    df["text"].str.contains(r"(°F[-+]$|\d-\s*$)", regex=True, na=False)
]

bad_sentences[["text"]]

In [ ]:
import re

def normalize_forecast_text(text: str) -> str:
    if not isinstance(text, str):
        return text

    # Remove markdown bold/italics
    text = re.sub(r"\*\*(.*?)\*\*", r"\1", text)

    # Remove standalone headers (lines with no sentence punctuation)
    text = re.sub(
        r"(?m)^(?:[A-Z][A-Za-z\s]+(?:\([^)]+\))?:)\s*$",
        "",
        text
    )

    # Convert bullet points to sentences
    text = re.sub(r"(?m)^-\s+", "", text)

    # Collapse excessive newlines
    text = re.sub(r"\n{2,}", " ", text)

    return text.strip()

In [ ]:
df["text"] = df["text"].apply(normalize_forecast_text)
df = sfe.split_into_sentences(sentence_col_name='clean sentence')
df

In [ ]:
# Return sentences that likely indicate bad sentence splits
bad_sentences = df[
    # Starts with dangling symbols or units (but not valid numbers)
    df["clean sentence"].str.match(r"^(°F|\+F|F\+|[^\w\d])", na=False)
    |
    # Ends mid temperature or numeric range expression
    df["clean sentence"].str.contains(r"(°F[-+]$|\d-\s*$)", regex=True, na=False)
]

bad_sentences[["clean sentence"]]

In [ ]:
bad_sentences